# Classification Tulipes / Lys

Pipeline mono-notebook : parsing -> prétraitement -> entraînement -> visualisation.

Un seul fichier Parquet est écrit, juste avant la visualisation.

> **Règles** : DataFrame uniquement · pas de `collect()` / `toPandas()` / `toList()` · Python pur = affichage seulement


## 0 · Session Spark & imports

In [5]:
import sys
import os
os.environ["HADOOP_HOME"] = "C:\\hadoop"
os.environ["PATH"] = os.environ["HADOOP_HOME"] + "\\bin;" + os.environ["PATH"]
import io
import struct

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, FloatType, ArrayType, StringType
)

print(sys.executable)


import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.model_selection import RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder
from scipy.stats import randint

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable


spark = (
    SparkSession.builder
    .appName("TulipsLilies")
    .config("spark.sql.files.ignoreCorruptFiles", "true")
    # Mémoire réduite : machine à 8 Go de RAM, éviter de saturer le système
    .config("spark.driver.memory", "1g")
    .config("spark.executor.memory", "1g")
    # Active un vrai traceback Python si l'UDF crash, au lieu d'une erreur Java opaque
    .config("spark.python.worker.faulthandler.enabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark version : {spark.version}")

c:\Users\Julien ANTOGNELLI\AppData\Local\Programs\Python\Python311\python.exe
Spark version : 4.1.1


## 1 - Chemins & constantes

In [6]:
TRAIN_PATH   = "./data/Train_5/"
TEST_PATH    = "./data/Test_5/"
OUTPUT_PREDS = "./output/predictions/"
MODEL_PATH   = "./output/model/"
TARGET_SIZE  = (64, 64)

In [7]:
print(os.environ.get("HADOOP_HOME"))
spark.read.format("binaryFile").load(TRAIN_PATH).limit(1).show()

C:\hadoop
+----+----------------+------+-------+
|path|modificationTime|length|content|
+----+----------------+------+-------+
+----+----------------+------+-------+



## 2 - Parsing

In [8]:
TARGET_W, TARGET_H = TARGET_SIZE

def decode_image_bytes(raw_bytes: bytes):
    try:
        from PIL import Image
        img = Image.open(io.BytesIO(raw_bytes)).convert("RGB")
        img = img.resize((TARGET_W, TARGET_H), Image.LANCZOS)
        raw = img.tobytes()
        n = len(raw)
        pixels = list(struct.unpack(f"{n}B", raw))
        return (TARGET_W, TARGET_H, 3, [float(p) for p in pixels])
    except Exception:
        return None

_decode_schema = StructType([
    StructField("width",    IntegerType(), False),
    StructField("height",   IntegerType(), False),
    StructField("channels", IntegerType(), False),
    StructField("pixels",   ArrayType(FloatType()), False),
])

decode_udf = F.udf(decode_image_bytes, _decode_schema)

def parse_images(path):
    raw = (
        spark.read.format("binaryFile")
        .option("recursiveFileLookup", "true")
        .option("pathGlobFilter", "*.{jpg,jpeg,png,JPG,PNG}")
        .load(path)
    )
    return (
        raw
        .coalesce(1)
        .select(
            F.regexp_extract(F.col("path"), r"([^/]+)$", 1).alias("image_id"),
            F.regexp_extract(F.col("path"), r"/([^/]+)/[^/]+$", 1).alias("label"),
            F.col("content").alias("raw_bytes"),
        )
        .withColumn("decoded", decode_udf(F.col("raw_bytes")))
        .filter(F.col("decoded").isNotNull())
        .select(
            "image_id", "label",
            F.col("decoded.pixels").alias("pixels"),
        )
    )

train_parsed_df = parse_images(TRAIN_PATH)
test_parsed_df  = parse_images(TEST_PATH)

print(f"Images train : {train_parsed_df.count()}")
print(f"Images test  : {test_parsed_df.count()}")
test_parsed_df.show()

Images train : 10
Images test  : 10
+----------+-------+--------------------+
|  image_id|  label|              pixels|
+----------+-------+--------------------+
|000139.jpg|tulipes|[180.0, 129.0, 79...|
|000137.jpg|tulipes|[184.0, 155.0, 50...|
|000063.jpg|    lys|[154.0, 2.0, 1.0,...|
|000140.jpg|tulipes|[118.0, 102.0, 84...|
|000061.jpg|    lys|[131.0, 119.0, 82...|
|000138.jpg|tulipes|[81.0, 15.0, 54.0...|
|000062.jpg|    lys|[205.0, 198.0, 16...|
|000064.jpg|    lys|[118.0, 145.0, 10...|
|000141.jpg|tulipes|[0.0, 72.0, 0.0, ...|
|000065.jpg|    lys|[164.0, 190.0, 21...|
+----------+-------+--------------------+



## 2.2 - Parsing: normalisation couleur

In [9]:
def normalize_rgb(pixels):
    if pixels is None:
        return None
    return [p / 255.0 for p in pixels]

normalize_rgb_udf = F.udf(normalize_rgb, ArrayType(FloatType()))

def preprocess_rgb(df):
    return df.withColumn("pixels_norm", normalize_rgb_udf(F.col("pixels")))

train_rgb_normalized_df  = preprocess_rgb(train_parsed_df)
test_rgb_normalized_df   = preprocess_rgb(test_parsed_df)

print("Aperçu après normalisation RGB :")
train_rgb_normalized_df.select("image_id", "label", "pixels_norm").show(5, truncate=40)

# Vérification rapide : taille attendue = 64*64*3 = 12288 valeurs RGB normalisées
expected_len = TARGET_W * TARGET_H * 3
check_len = (
    train_rgb_normalized_df
    .select(F.size(F.col("pixels_norm")).alias("len"))
    .first()["len"]
)
print(f"Taille pixels_norm (attendu {expected_len}) : {check_len}")

Aperçu après normalisation RGB :
+----------+-------+----------------------------------------+
|  image_id|  label|                             pixels_norm|
+----------+-------+----------------------------------------+
|000005.jpg|    lys|[0.101960786, 0.14117648, 0.05882353,...|
|000004.jpg|tulipes|[0.39607844, 0.37254903, 0.25490198, ...|
|000004.jpg|    lys|[0.78431374, 0.81960785, 0.85490197, ...|
|000002.jpg|    lys|[0.5568628, 0.5137255, 0.54901963, 0....|
|000001.jpg|    lys|[0.078431375, 0.105882354, 0.01960784...|
+----------+-------+----------------------------------------+
only showing top 5 rows
Taille pixels_norm (attendu 12288) : 12288


## 3 - Prétraitement 

On garde `pixels` (RGB brut, 0-255) intact pour l'affichage futur dans Streamlit,
et on ajoute une colonne `pixels_gray` : niveaux de gris normalisés en [0.0, 1.0].

Conversion RGB -> nuances de gris : formule de luminance pondérée (standard) :
```
gray = 0.299*R + 0.587*G + 0.114*B
```

`pixels` est une liste aplatie `[R,G,B, R,G,B, ...]` de taille 64*64*3 = 12288.
L'UDF regroupe les valeurs par 3, applique la formule et renvoie les pixels en niveaux de gris

In [10]:
def rgb_to_grayscale(pixels):
    if pixels is None:
        return None
    gray = []
    for i in range(0, len(pixels), 3):
        r, g, b = pixels[i], pixels[i + 1], pixels[i + 2]
        # formule standard de luminance perceptuelle
        g_value = 0.299 * r + 0.587 * g + 0.114 * b
        gray.append(g_value)
    return gray

gray_udf = F.udf(rgb_to_grayscale, ArrayType(FloatType()))

def preprocess(df):
    return df.withColumn("pixels_grayscale", gray_udf(F.col("pixels")))

train_preprocessed_gray_df = preprocess(train_parsed_df)
test_preprocessed_gray_df  = preprocess(test_parsed_df)

print("Aperçu après prétraitement :")
train_preprocessed_gray_df.select("image_id", "label", "pixels_grayscale").show(5, truncate=40)

# Vérification rapide : taille attendue = 64*64 = 4096 valeurs en niveaux de gris
expected_len = TARGET_W * TARGET_H
check_len = (
    train_preprocessed_gray_df
    .select(F.size(F.col("pixels_grayscale")).alias("len"))
    .first()["len"]
)
print(f"Taille pixels_grayscale (attendu {expected_len}) : {check_len}")

Aperçu après prétraitement :
+----------+-------+----------------------------------------+
|  image_id|  label|                        pixels_grayscale|
+----------+-------+----------------------------------------+
|000005.jpg|    lys|[30.616, 32.018, 33.491, 35.263, 37.9...|
|000004.jpg|tulipes|[93.374, 96.787, 99.684, 102.755, 104...|
|000004.jpg|    lys|[207.335, 205.221, 205.107, 205.107, ...|
|000002.jpg|    lys|[135.315, 136.201, 135.087, 136.087, ...|
|000001.jpg|    lys|[22.399, 18.274, 60.137, 102.701, 108...|
+----------+-------+----------------------------------------+
only showing top 5 rows
Taille pixels_grayscale (attendu 4096) : 4096


In [11]:
def rgb_to_normalized_gray(pixels):
    if pixels is None:
        return None
    gray = []
    for i in range(0, len(pixels), 3):
        r, g, b = pixels[i], pixels[i + 1], pixels[i + 2]
        # formule standard de luminance perceptuelle
        g_value = 0.299 * r + 0.587 * g + 0.114 * b
        # normalise la valeur résultante dans l'intervalle [0, 1] au lieu de [0, 255],
        gray.append(g_value / 255.0)
    return gray

gray_udf = F.udf(rgb_to_normalized_gray, ArrayType(FloatType()))

def preprocess(df):
    return df.withColumn("pixels_gray", gray_udf(F.col("pixels")))

train_preprocessed_norm_gray_df = preprocess(train_parsed_df)
test_preprocessed_norm_gray_df  = preprocess(test_parsed_df)

print("Aperçu après prétraitement :")
train_preprocessed_norm_gray_df.select("image_id", "label", "pixels_gray").show(5, truncate=40)

# Vérification rapide : taille attendue = 64*64 = 4096 valeurs en niveaux de gris
expected_len = TARGET_W * TARGET_H
check_len = (
    train_preprocessed_norm_gray_df
    .select(F.size(F.col("pixels_gray")).alias("len"))
    .first()["len"]
)
print(f"Taille pixels_gray (attendu {expected_len}) : {check_len}")

Aperçu après prétraitement :
+----------+-------+----------------------------------------+
|  image_id|  label|                             pixels_gray|
+----------+-------+----------------------------------------+
|000005.jpg|    lys|[0.120062746, 0.12556079, 0.13133726,...|
|000004.jpg|tulipes|[0.36617255, 0.37955686, 0.39091766, ...|
|000004.jpg|    lys|[0.8130784, 0.80478823, 0.8043412, 0....|
|000002.jpg|    lys|[0.53064704, 0.5341216, 0.52975297, 0...|
|000001.jpg|    lys|[0.087839216, 0.07166275, 0.23583138,...|
+----------+-------+----------------------------------------+
only showing top 5 rows
Taille pixels_gray (attendu 4096) : 4096


## ML couleurs - prétraitement en bytes

In [20]:
# Prétraitement couleur (bytes) : RGB brut 0-255, encodé en BinaryType
from pyspark.sql.types import BinaryType

def pixels_to_bytes(pixels):
    if pixels is None:
        return None
    int_pixels = [int(p) for p in pixels]
    return struct.pack(f"{len(int_pixels)}B", *int_pixels)

bytes_udf = F.udf(pixels_to_bytes, BinaryType())

def preprocess_color_bytes(df):
    return df.withColumn("pixels_color_bytes", bytes_udf(F.col("pixels")))

train_color_bytes_df = preprocess_color_bytes(train_parsed_df)
test_color_bytes_df  = preprocess_color_bytes(test_parsed_df)

print("Aperçu après prétraitement :")
train_color_bytes_df.select("image_id", "label", "pixels_color_bytes").show(5, truncate=40)

expected_len = TARGET_W * TARGET_H * 3
check_len = (
    train_color_bytes_df
    .select(F.length(F.col("pixels_color_bytes")).alias("len"))
    .first()["len"]
)
print(f"Taille pixels_color_bytes en octets (attendu {expected_len}) : {check_len}")

Aperçu après prétraitement :
+----------+-------+----------------------------------------+
|  image_id|  label|                      pixels_color_bytes|
+----------+-------+----------------------------------------+
|000005.jpg|    lys|[1A 24 0F 1A 26 11 1B 28 11 1D 2A 11 ...|
|000004.jpg|tulipes|[65 5F 41 69 62 45 6E 64 47 72 67 48 ...|
|000004.jpg|    lys|[C8 D1 DA C6 CF D7 C6 CF D6 C6 CF D6 ...|
|000002.jpg|    lys|[8E 83 8C 8F 84 8C 8E 83 8A 8F 84 8B ...|
|000001.jpg|    lys|[14 1B 05 0E 18 00 3A 3F 33 66 67 67 ...|
+----------+-------+----------------------------------------+
only showing top 5 rows
Taille pixels_color_bytes en octets (attendu 12288) : 12288


ValueError: X has 4096 features, but RandomForestClassifier is expecting 12288 features as input.

## ML couleurs normalisées


In [13]:
spark = SparkSession.builder \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "false") \
    .config("spark.python.worker.memory", "2g") \
    .getOrCreate()

In [14]:
FEATURE_COL = "pixels_norm"
LABEL_COL   = "label"

def to_numpy(df, feature_col=FEATURE_COL, label_col=LABEL_COL):
    # Ramène (image_id, label, features) du cluster vers le driver
    # Seul point de collect() du pipeline ML — exception documentée
    rows = df.select("image_id", label_col, feature_col).collect()
    image_ids = [r["image_id"] for r in rows]
    X = np.array([r[feature_col] for r in rows], dtype=np.float32)
    y = [r[label_col] for r in rows]
    return image_ids, X, y

# Collecte train / test (driver)
train_ids, X_train, y_train_raw = to_numpy(train_rgb_normalized_df)
test_ids,  X_test,  y_test_raw  = to_numpy(test_rgb_normalized_df)

print(f"X_train shape : {X_train.shape}")
print(f"X_test  shape : {X_test.shape}")

# --- Encodage des labels (lys/tulipes -> 0/1) ---
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train_raw)
y_test  = label_encoder.transform(y_test_raw)
print(f"Classes : {dict(enumerate(label_encoder.classes_))}")

# --- Entraînement RandomForest ---
rf_rgb_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    random_state=42,
    n_jobs=-1,
)
rf_rgb_model.fit(X_train, y_train)

# --- Évaluation ---
y_pred = rf_rgb_model.predict(X_test)
y_pred_proba = rf_rgb_model.predict_proba(X_test)

acc = accuracy_score(y_test, y_pred)
print(f"\nAccuracy (couleurs normalisées) : {acc:.4f}")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

X_train shape : (10, 12288)
X_test  shape : (10, 12288)
Classes : {0: 'lys', 1: 'tulipes'}

Accuracy (couleurs normalisées) : 0.4000
              precision    recall  f1-score   support

         lys       0.43      0.60      0.50         5
     tulipes       0.33      0.20      0.25         5

    accuracy                           0.40        10
   macro avg       0.38      0.40      0.38        10
weighted avg       0.38      0.40      0.38        10



In [15]:
train_rgb_normalized_df.count()

10

In [16]:
train_rgb_normalized_df.select("pixels_norm").show(1, truncate=50)

+--------------------------------------------------+
|                                       pixels_norm|
+--------------------------------------------------+
|[0.101960786, 0.14117648, 0.05882353, 0.1019607...|
+--------------------------------------------------+
only showing top 1 row


In [17]:
from pyspark.sql import functions as F
train_rgb_normalized_df.select(F.size("pixels_norm")).show(10)

+-----------------+
|size(pixels_norm)|
+-----------------+
|            12288|
|            12288|
|            12288|
|            12288|
|            12288|
|            12288|
|            12288|
|            12288|
|            12288|
|            12288|
+-----------------+



## ML grayscale

In [18]:
## 4.1 - ML : Grayscale non normalisé (0-255)

LABEL_COL   = "label"
FEATURE_COL = "pixels_grayscale"

def to_numpy(df, feature_col, label_col=LABEL_COL):
    rows = df.select("image_id", label_col, feature_col).collect()
    image_ids = [r["image_id"]    for r in rows]
    X         = np.array([r[feature_col] for r in rows], dtype=np.float32)
    y         = [r[label_col]     for r in rows]
    return image_ids, X, y

train_ids, X_train, y_train_raw = to_numpy(train_preprocessed_gray_df, FEATURE_COL)
test_ids,  X_test,  y_test_raw  = to_numpy(test_preprocessed_gray_df,  FEATURE_COL)

print(f"X_train shape : {X_train.shape}")
print(f"X_test  shape : {X_test.shape}")

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train_raw)
y_test  = label_encoder.transform(y_test_raw)
print(f"Classes : {dict(enumerate(label_encoder.classes_))}")

rf_gray_raw = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_gray_raw.fit(X_train, y_train)

y_pred       = rf_gray_raw.predict(X_test)
y_pred_proba = rf_gray_raw.predict_proba(X_test)

print(f"\nAccuracy (grayscale non normalisé) : {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

predicted_labels = label_encoder.inverse_transform(y_pred)
confidences      = y_pred_proba.max(axis=1).tolist()

predictions_gray_raw_df = spark.createDataFrame(
    list(zip(test_ids, y_test_raw, predicted_labels.tolist(), confidences)),
    schema=StructType([
        StructField("image_id",        StringType(), False),
        StructField("true_label",      StringType(), False),
        StructField("predicted_label", StringType(), False),
        StructField("confidence",      FloatType(),  False),
    ]),
)

print("\nPrédictions (Spark DataFrame) :")
predictions_gray_raw_df.show(10, truncate=False)

X_train shape : (10, 4096)
X_test  shape : (10, 4096)
Classes : {0: 'lys', 1: 'tulipes'}

Accuracy (grayscale non normalisé) : 0.7000
              precision    recall  f1-score   support

         lys       0.75      0.60      0.67         5
     tulipes       0.67      0.80      0.73         5

    accuracy                           0.70        10
   macro avg       0.71      0.70      0.70        10
weighted avg       0.71      0.70      0.70        10


Prédictions (Spark DataFrame) :
+----------+----------+---------------+----------+
|image_id  |true_label|predicted_label|confidence|
+----------+----------+---------------+----------+
|000139.jpg|tulipes   |tulipes        |0.58      |
|000137.jpg|tulipes   |tulipes        |0.535     |
|000063.jpg|lys       |tulipes        |0.615     |
|000140.jpg|tulipes   |lys            |0.6       |
|000061.jpg|lys       |tulipes        |0.715     |
|000138.jpg|tulipes   |tulipes        |0.535     |
|000062.jpg|lys       |lys            |0.57    

## ML grayscale normalisé

In [19]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
import numpy as np

LABEL_COL   = "label"
FEATURE_COL = "pixels_gray"

def to_numpy(df, feature_col=FEATURE_COL, label_col=LABEL_COL):
    rows = df.select("image_id", label_col, feature_col).collect()
    image_ids = [r["image_id"] for r in rows]
    X = np.array([r[feature_col] for r in rows], dtype=np.float32)
    y = [r[label_col] for r in rows]
    return image_ids, X, y

train_ids, X_train_gray, y_train_raw = to_numpy(train_preprocessed_norm_gray_df)
test_ids,  X_test_gray,  y_test_raw  = to_numpy(test_preprocessed_norm_gray_df)

print(f"X_train_gray shape : {X_train_gray.shape}")
print(f"X_test_gray  shape : {X_test_gray.shape}")

label_encoder = LabelEncoder()
label_encoder.fit(y_train_raw)
y_train = label_encoder.transform(y_train_raw)
y_test  = label_encoder.transform(y_test_raw)
print(f"Classes : {dict(enumerate(label_encoder.classes_))}")

rf_gray_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    random_state=42,
    n_jobs=-1,
)
rf_gray_model.fit(X_train_gray, y_train)

y_pred_gray       = rf_gray_model.predict(X_test_gray)
y_pred_proba_gray = rf_gray_model.predict_proba(X_test_gray)

acc_gray = accuracy_score(y_test, y_pred_gray)
print(f"\nAccuracy (grayscale normalisé) : {acc_gray:.4f}")
print(classification_report(y_test, y_pred_gray, target_names=label_encoder.classes_))

predicted_labels_gray = label_encoder.inverse_transform(y_pred_gray)
confidences_gray      = y_pred_proba_gray.max(axis=1).tolist()

predictions_gray_df = spark.createDataFrame(
    list(zip(test_ids, y_test_raw, predicted_labels_gray.tolist(), confidences_gray)),
    schema=StructType([
        StructField("image_id",        StringType(), False),
        StructField("true_label",      StringType(), False),
        StructField("predicted_label", StringType(), False),
        StructField("confidence",      FloatType(),  False),
    ]),
)

print("\nPrédictions grayscale (Spark DataFrame) :")
predictions_gray_df.show(10, truncate=False)

X_train_gray shape : (10, 4096)
X_test_gray  shape : (10, 4096)
Classes : {0: 'lys', 1: 'tulipes'}

Accuracy (grayscale normalisé) : 0.7000
              precision    recall  f1-score   support

         lys       0.75      0.60      0.67         5
     tulipes       0.67      0.80      0.73         5

    accuracy                           0.70        10
   macro avg       0.71      0.70      0.70        10
weighted avg       0.71      0.70      0.70        10


Prédictions grayscale (Spark DataFrame) :
+----------+----------+---------------+----------+
|image_id  |true_label|predicted_label|confidence|
+----------+----------+---------------+----------+
|000139.jpg|tulipes   |tulipes        |0.58      |
|000137.jpg|tulipes   |tulipes        |0.535     |
|000063.jpg|lys       |tulipes        |0.615     |
|000140.jpg|tulipes   |lys            |0.6       |
|000061.jpg|lys       |tulipes        |0.715     |
|000138.jpg|tulipes   |tulipes        |0.535     |
|000062.jpg|lys       |lys     